# Session 4 — Weights, Systematics, Fitting, and Limits

In the final session we treat the bb + MET selection as a **full analysis**: we include event weights and simple systematic uncertainties, perform a binned likelihood fit of signal + backgrounds to data, assess goodness of fit, and derive an upper limit on the signal strength. This mirrors, in a simplified way, the statistical treatment used in the CMS bb + MET JHEP result.

**Learning objectives:**
- Understand event weights and systematic uncertainties in a bb + MET analysis
- Perform a binned likelihood fit of signal + backgrounds to data in MET bins
- Assess goodness of fit (χ², p-values, pulls)
- Compute an upper limit on signal strength for the bbMET signal

**Input:** This session uses the processor output (e.g. `output_2017.pkl`) from Session 3. Run `python run_analysis.py` or `run_analysis.py --full` first if you do not have the pickle file.

![Placeholder: schematic limit plot (signal strength vs mediator mass or tanβ)](figures/session4_fig_limits_placeholder.png)

---
## 0. Load processor output

Load processor histograms and cutflows. After `run_analysis.py` (default), each key is a **merged physics group** (`data`, `DYJets`, `ZJets`, `WJets`, `Top`, `STop`, `DIBOSON`, `SMH`) plus split signal keys for randomized signal (`signal_mA600_ma*_mchi1`). Each value is an accumulator with `recoil`, `cutflow`, etc. Use `--per-dataset` on the runner for one key per YAML sample (legacy).

In [ ]:
# Session 4 setup: load output bundle, observable templates, and systematics
import sys, os
try:
    sys.setrecursionlimit(8000)
except Exception:
    pass

import pickle
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep
from scipy.optimize import minimize
from scipy.stats import poisson, chi2

hep.style.use("CMS")

# Ensure project root is importable
for _p in (".", ".."):
    if os.path.isfile(os.path.join(_p, "config", "datasets_2017.py")):
        sys.path.insert(0, os.path.abspath(_p))
        break

from config.datasets_2017 import data_and_bkg_keys


def _load_results_bundle():
    candidates = [
        "output/output_2017.pkl", "output/output_2017_full.pkl",
        "output_2017.pkl", "output_2017_full.pkl",
        "../output/output_2017.pkl", "../output/output_2017_full.pkl",
    ]
    for path in candidates:
        if os.path.isfile(path):
            with open(path, "rb") as f:
                return pickle.load(f)
    raise FileNotFoundError("Run run_analysis.py (or condor merge) first to produce output pickle.")


bundle_or_results = _load_results_bundle()
if isinstance(bundle_or_results, dict) and isinstance(bundle_or_results.get("samples"), dict):
    metadata = bundle_or_results.get("metadata", {})
    results = bundle_or_results["samples"]
else:
    metadata = {}
    results = bundle_or_results


def _unwrap_hist_obj(obj):
    return obj._hist if hasattr(obj, "_hist") else obj


def _pick_observable_and_hist(sample_acc):
    # Prefer SR cos(theta*) if available, else SR recoil.
    hbr = sample_acc.get("hists_by_region", {}) if isinstance(sample_acc, dict) else {}
    if isinstance(hbr, dict):
        cts = _unwrap_hist_obj(hbr.get("cos_theta_star"))
        if cts is not None:
            try:
                return "cos_theta_star", cts[{"region": "sr"}]
            except Exception:
                pass
        recoil = _unwrap_hist_obj(hbr.get("recoil"))
        if recoil is not None:
            try:
                return "recoil", recoil[{"region": "sr"}]
            except Exception:
                pass

    # Legacy flat fallback
    cts = _unwrap_hist_obj(sample_acc.get("cos_theta_star"))
    if cts is not None:
        return "cos_theta_star", cts
    recoil = _unwrap_hist_obj(sample_acc.get("recoil"))
    if recoil is not None:
        return "recoil", recoil
    raise KeyError("No compatible SR observable histogram found (cos_theta_star/recoil).")


def _hist_vals_vars(h):
    vals = np.asarray(h.values(), dtype=float).flatten()
    vars_ = h.variances()
    if vars_ is None:
        vars_ = np.clip(vals, 0.0, None)
    else:
        vars_ = np.clip(np.asarray(vars_, dtype=float).flatten(), 0.0, None)
    edges = np.asarray(h.axes[0].edges, dtype=float)
    return vals, vars_, edges


data_datasets, bkg_datasets = data_and_bkg_keys(results)
signal_datasets = sorted([k for k in results.keys() if str(k).startswith("signal_")])
print("Data keys:", data_datasets)
print("Background keys:", bkg_datasets)
print("Signal keys:", signal_datasets[:6], "..." if len(signal_datasets) > 6 else "")

# Build observed data histogram and per-process background templates.
obs = None
obs_edges = None
for d in data_datasets:
    if d not in results:
        continue
    obs_name, h = _pick_observable_and_hist(results[d])
    vals, _, edges = _hist_vals_vars(h)
    if obs is None:
        obs = vals.copy()
        obs_edges = edges
        observable_name = obs_name
    else:
        if len(vals) == len(obs):
            obs += vals

bkg_by_process = {}
bkg_var_by_process = {}
for b in bkg_datasets:
    if b not in results:
        continue
    _, h = _pick_observable_and_hist(results[b])
    vals, vars_, edges = _hist_vals_vars(h)
    if obs is not None and len(vals) != len(obs):
        continue
    bkg_by_process[b] = vals
    bkg_var_by_process[b] = vars_

if obs is None:
    if bkg_by_process:
        # fallback if no data available
        first = next(iter(bkg_by_process.values()))
        obs = np.zeros_like(first)
        obs_edges = np.arange(len(first) + 1, dtype=float)
    else:
        obs = np.zeros(10, dtype=float)
        obs_edges = np.arange(11, dtype=float)

bkg = np.sum(np.vstack(list(bkg_by_process.values())), axis=0) if bkg_by_process else np.ones_like(obs)
bkg_stat_var = np.sum(np.vstack(list(bkg_var_by_process.values())), axis=0) if bkg_var_by_process else np.clip(bkg, 0.0, None)

# Signal template: sum available MH3=600 mass points (for simple limit exercise)
sig = None
for s in signal_datasets:
    _, h = _pick_observable_and_hist(results[s])
    vals, _, _ = _hist_vals_vars(h)
    if sig is None:
        sig = vals.copy()
    elif len(vals) == len(sig):
        sig += vals
if sig is None:
    sig = np.zeros_like(bkg)

# Read systematics from output metadata (with safe defaults)
sys_cfg = metadata.get("systematics", {}) if isinstance(metadata, dict) else {}
lumi_rel_unc = float(sys_cfg.get("lumi_rel_unc", 0.025))
default_bkg_norm = float(sys_cfg.get("default_bkg_norm_rel_unc", 0.10))
shape_map = sys_cfg.get("shape_rel_unc_by_observable", {}) if isinstance(sys_cfg, dict) else {}
shape_rel_unc = float(shape_map.get(observable_name, 0.05))
proc_map = sys_cfg.get("bkg_norm_rel_unc_by_process", {}) if isinstance(sys_cfg, dict) else {}

# Build per-bin syst variance from process normalization nuisances (uncorrelated by process)
proc_syst_var = np.zeros_like(bkg)
for pname, pvals in bkg_by_process.items():
    rel = float(proc_map.get(pname, default_bkg_norm))
    proc_syst_var += (rel * np.clip(pvals, 0.0, None)) ** 2

lumi_syst_var = (lumi_rel_unc * np.clip(bkg, 0.0, None)) ** 2
shape_syst_var = (shape_rel_unc * np.clip(bkg, 0.0, None)) ** 2

bkg_syst_var = proc_syst_var + lumi_syst_var + shape_syst_var
bkg_total_var = bkg_stat_var + bkg_syst_var
bkg_total_sigma = np.sqrt(np.clip(bkg_total_var, 0.0, None))

# Effective global fractional uncertainty for a simple constrained-nuisance fit model.
frac_unc_eff = float(np.sqrt(np.sum(bkg_syst_var)) / max(np.sum(bkg), 1e-9))

print(f"Observable in SR: {observable_name}")
print(f"Data sum = {obs.sum():.2f}, Bkg sum = {bkg.sum():.2f}, Sig(sum of signal_*) = {sig.sum():.2f}")
print(f"Systematics: lumi={lumi_rel_unc:.3f}, shape({observable_name})={shape_rel_unc:.3f}, eff_global={frac_unc_eff:.3f}")


---
## 1. Weights and systematics

The processor uses **genWeight** (sign only for MC) when filling histograms. In addition, pickles produced by `run_analysis.py --full` are already luminosity-normalised for MC (using YAML cross sections), so do **not** apply an extra xsec×lumi scaling in this notebook.

**Systematic uncertainties** affect the predicted yields and shapes:
- **Normalisation:** e.g. ±20% per process (theory, luminosity).
- **Shape:** e.g. MET scale, b-tag efficiency, jet energy scale.

Here we introduce a simple **normalisation systematic** per background: we scale each histogram by a factor \((1 + \alpha \cdot \sigma)\), where \(\sigma\) is the relative uncertainty (e.g. 0.2) and \(\alpha\) is a nuisance parameter (fitted or fixed at ±1 for up/down).

In [ ]:
# Background-only fit with one normalization nuisance constrained by systematics
# Model: lambda_i(beta) = beta * bkg_i, with Gaussian prior beta ~ N(1, frac_unc_eff)
# If frac_unc_eff is tiny, keep a minimum to avoid singular fit.
sigma_beta = max(frac_unc_eff, 1e-3)


def nll_bonly(pars):
    beta = float(pars[0])
    lam = np.clip(beta * bkg, 1e-9, None)
    nll_pois = -np.sum(poisson.logpmf(obs.astype(int), lam))
    nll_constr = 0.5 * ((beta - 1.0) / sigma_beta) ** 2
    return nll_pois + nll_constr

fit = minimize(nll_bonly, x0=[1.0], bounds=[(0.01, 5.0)])
beta_best = float(fit.x[0])
expected = beta_best * bkg
expected_sigma = beta_best * bkg_total_sigma

print(f"Best-fit beta = {beta_best:.4f}  (sigma_beta prior = {sigma_beta:.4f})")
print(f"Post-fit expected sum = {expected.sum():.2f} vs observed sum = {obs.sum():.2f}")


---
## 2. Binned fit

We fit a **binned likelihood**: in each MET bin \(i\), the observed count \(n_i\) is Poisson with mean \(\mu \cdot s_i + \sum_b b_{i,b}\), where \(s_i\) is the signal prediction, \(b_{i,b}\) the background \(b\) in bin \(i\), and \(\mu\) is the signal strength (1 = SM signal, 0 = background only).

**Simplified:** Assume only backgrounds (no signal histogram yet). Fit the sum of background histograms to the data histogram by scaling backgrounds with normalisation factors (or fix them and compute \(\chi^2\)). If you have a signal histogram, add \(\mu \cdot s\) and fit \(\mu\).

In [ ]:
# Goodness-of-fit and CMS-style post-fit plot (main + ratio + pulls)
chi2_val = np.sum((obs - expected) ** 2 / np.clip(expected + expected_sigma**2, 1e-9, None))
ndof = max(len(obs) - 1, 1)
pvalue = 1.0 - chi2.cdf(chi2_val, ndof)
print(f"GOF: chi2 = {chi2_val:.2f}, ndof = {ndof}, p-value = {pvalue:.4f}")

centers = 0.5 * (obs_edges[:-1] + obs_edges[1:])
widths = np.diff(obs_edges)

fig, (ax, rax) = plt.subplots(
    2, 1, sharex=True, figsize=(7.2, 6.4), gridspec_kw={"height_ratios": [3.0, 1.0], "hspace": 0.05}
)

# Main: post-fit background + total unc band + observed points
ax.bar(centers, expected, width=widths, color="#2A64AD", alpha=0.85, edgecolor="none", label="Post-fit bkg")
ax.fill_between(
    obs_edges,
    np.r_[np.clip(expected - expected_sigma, 0.0, None), np.clip(expected - expected_sigma, 0.0, None)[-1]],
    np.r_[expected + expected_sigma, (expected + expected_sigma)[-1]],
    step="post",
    facecolor="none",
    edgecolor="#b22222",
    hatch="xx",
    linewidth=0.0,
    alpha=0.7,
    label="Stat. + Syst. unc.",
)
obs_err = np.sqrt(np.clip(obs, 0.0, None))
ax.errorbar(centers, obs, yerr=obs_err, fmt="o", color="black", markersize=4, linewidth=1.0, label="Data", zorder=10)
ax.set_yscale("log")
ymin, ymax = ax.get_ylim()
ax.set_ylim(max(1e-3, ymin), max(1.0, ymax) * 30.0)
ax.set_ylabel("Events")
hep.cms.label("Preliminary", data=True, lumi=float(metadata.get("lumi_fb", 41.5)), year=int(metadata.get("year", 2017)), ax=ax)
ax.legend(loc="upper right", fontsize=8.5, frameon=False)
ax.grid(alpha=0.18, axis="y")

# Ratio panel: Data/Pred and uncertainty band
ratio = np.divide(obs, expected, out=np.full_like(obs, np.nan, dtype=float), where=expected > 0)
ratio_err = np.divide(obs_err, expected, out=np.full_like(obs_err, np.nan, dtype=float), where=expected > 0)
unc_ratio = np.divide(expected_sigma, expected, out=np.full_like(expected_sigma, np.nan, dtype=float), where=expected > 0)
rax.fill_between(
    obs_edges,
    np.r_[1.0 - unc_ratio, (1.0 - unc_ratio)[-1]],
    np.r_[1.0 + unc_ratio, (1.0 + unc_ratio)[-1]],
    step="post",
    facecolor="none",
    edgecolor="#b22222",
    hatch="xx",
    linewidth=0.0,
    alpha=0.7,
    label="Stat. + Syst. unc.",
)
rax.errorbar(centers, ratio, yerr=ratio_err, fmt="o", color="black", markersize=4, linewidth=1.0)
rax.axhline(1.0, color="black", lw=1.0)
rax.set_ylim(0.4, 1.6)
rax.set_ylabel("Data/Pred.")
rax.set_xlabel(observable_name)
rax.legend(loc="upper right", fontsize=8, frameon=False)
rax.grid(alpha=0.18, axis="y")

# Additional pull plot
pulls = np.divide(obs - expected, np.sqrt(np.clip(expected + expected_sigma**2, 1e-9, None)))
fig2, ax2 = plt.subplots(figsize=(7.2, 2.8))
ax2.bar(np.arange(len(pulls)), pulls, color="#6E6E6E", edgecolor="black", alpha=0.8)
ax2.axhline(0.0, color="black", ls="-")
ax2.axhline(1.0, color="gray", ls="--", lw=1.0)
ax2.axhline(-1.0, color="gray", ls="--", lw=1.0)
ax2.set_xlabel("Bin")
ax2.set_ylabel("Pull")
ax2.set_title("Pull distribution (post-fit)")
ax2.grid(alpha=0.18, axis="y")

plt.show()


---
## 3. Goodness of fit

After the fit, check how well the model describes the data:
- **\(\chi^2\):** \(\chi^2 = \sum_i (n_i - \lambda_i)^2 / \lambda_i\) (or use variance \(\lambda_i\) for Poisson).
- **Degrees of freedom:** number of bins minus number of fitted parameters.
- **p-value:** \(P(\chi^2 \geq \chi^2_{\text{obs}})\) from the \(\chi^2\) distribution.
- **Pulls:** \((n_i - \lambda_i) / \sigma_i\) per bin to spot which bins disagree.

In [ ]:
# Approximate 95% CL limit from profile-likelihood scan in signal strength mu
# Model: lambda_i = beta*bkg_i + mu*sig_i, with beta constrained by systematics prior.

sigma_beta = max(frac_unc_eff, 1e-3)


def prof_nll(mu):
    mu = float(mu)

    def nll_beta(beta_arr):
        beta = float(beta_arr[0])
        lam = np.clip(beta * bkg + mu * sig, 1e-9, None)
        nll_p = -np.sum(poisson.logpmf(obs.astype(int), lam))
        nll_c = 0.5 * ((beta - 1.0) / sigma_beta) ** 2
        return nll_p + nll_c

    res = minimize(nll_beta, x0=[beta_best], bounds=[(0.01, 5.0)])
    return float(res.fun), float(res.x[0])

# Global best (mu >= 0)
mu_grid = np.linspace(0.0, 5.0, 101)
scan_vals = []
scan_beta = []
for mu in mu_grid:
    v, bfit = prof_nll(mu)
    scan_vals.append(v)
    scan_beta.append(bfit)
scan_vals = np.asarray(scan_vals, dtype=float)
scan_beta = np.asarray(scan_beta, dtype=float)

imin = int(np.argmin(scan_vals))
mu_hat = float(mu_grid[imin])
nll_min = float(scan_vals[imin])
qmu = 2.0 * (scan_vals - nll_min)

# 95% CL (1 dof approximation): q_mu = 2.71 crossing on the right of mu_hat
target = 2.71
mask = mu_grid >= mu_hat
mu95 = np.nan
if np.any(mask):
    x = mu_grid[mask]
    y = qmu[mask]
    idx = np.where(y >= target)[0]
    if len(idx) > 0:
        j = int(idx[0])
        if j == 0:
            mu95 = float(x[0])
        else:
            x0, x1 = x[j - 1], x[j]
            y0, y1 = y[j - 1], y[j]
            mu95 = float(x0 + (target - y0) * (x1 - x0) / max(y1 - y0, 1e-9))

print(f"mu_hat = {mu_hat:.3f}")
print(f"Approx. 95% CL upper limit on signal strength mu: {mu95:.3f}")

# CMS-like limit scan plot
fig, ax = plt.subplots(figsize=(7.0, 4.8))
ax.plot(mu_grid, qmu, color="#2A64AD", lw=2.0, label=r"$q_\mu$ scan")
ax.axhline(2.71, color="#b22222", ls="--", lw=1.6, label="95% CL threshold")
ax.axvline(mu_hat, color="black", ls=":", lw=1.3, label=fr"$\hat{{\mu}}={mu_hat:.2f}$")
if np.isfinite(mu95):
    ax.axvline(mu95, color="#b22222", ls="-.", lw=1.5, label=fr"$\mu_{{95}}={mu95:.2f}$")
ax.set_xlabel(r"Signal strength $\mu$")
ax.set_ylabel(r"$q_\mu = -2\ln\lambda(\mu)$")
ax.set_title("Approximate profile-likelihood limit scan")
hep.cms.label("Preliminary", data=True, lumi=float(metadata.get("lumi_fb", 41.5)), year=int(metadata.get("year", 2017)), ax=ax)
ax.grid(alpha=0.2)
ax.legend(frameon=False)
plt.show()


---
## 4. Limit calculation

An **upper limit** on the signal strength \(\mu\) at e.g. 95% CL says: we exclude \(\mu > \mu_{\text{limit}}\) at 95% confidence. Common methods:
- **Asymptotic formula:** Use the profile likelihood ratio and the asymptotic distribution (e.g. \(2\Delta\ln L\) is chi-squared distributed).
- **pyhf:** The [pyhf](https://pyhf.readthedocs.io/) library implements histogram fits and asymptotic limits (and toys if needed).

**Simplified exercise:** For a single bin (e.g. total SR yield), the 95% CL upper limit on the number of signal events can be computed with the one-sided Poisson formula (e.g. \(n_{\text{obs}}\) observed, \(b\) expected background → upper limit on \(s\)). Or use scipy to find \(\mu\) such that the profile likelihood (or a simple likelihood) gives the desired CL.

In [ ]:
# Fit summary table (quick diagnostic)
summary = {
    "observable": observable_name,
    "n_bins": int(len(obs)),
    "obs_total": float(obs.sum()),
    "bkg_prefit_total": float(bkg.sum()),
    "bkg_postfit_total": float(expected.sum()),
    "signal_total": float(sig.sum()),
    "beta_best": float(beta_best),
    "sigma_beta_prior": float(sigma_beta),
    "gof_chi2": float(chi2_val),
    "gof_ndof": int(ndof),
    "gof_pvalue": float(pvalue),
    "mu_hat": float(mu_hat),
    "mu95": float(mu95) if np.isfinite(mu95) else None,
}
for k, v in summary.items():
    print(f"{k:18s}: {v}")


---
## 5. Optional: Combine datacards, GoF, and limits for all mass points

For a simple Combine workflow from the merged pickle output, use the helper scripts added in `scripts/`:

- `scripts/make_combine_datacards.py` builds one counting datacard per signal mass point in `output/combine_cards`.
- `output/combine_cards/run_combine_all.sh` runs `GoodnessOfFit` and `AsymptoticLimits` for all datacards.
- `scripts/plot_combine_limits.py` makes one expected-limit plot in `output/combine_plots`.

> Run the Combine commands from a CMSSW environment where `combine` is available.

In [ ]:
# Build simple datacards from output pickle and print Combine commands
import os
import subprocess
from pathlib import Path

# Prefer canonical EOS path: /eos/user/<firstletter>/<username>/SWAN_projects/bbdm_cmsdas26
user = os.environ.get("USER", "").strip()
preferred_repo = (
    Path(f"/eos/user/{user[:1]}/{user}/SWAN_projects/bbdm_cmsdas26")
    if user
    else None
)

if preferred_repo is not None and (preferred_repo / "run_analysis.py").exists():
    repo = preferred_repo
else:
    repo = Path.cwd()
    if (repo / "run_analysis.py").exists() is False and (repo.parent / "run_analysis.py").exists():
        repo = repo.parent

print("Using repository:", repo)

pkl_path = repo / "output" / "output_2017_full.pkl"
cards_dir = repo / "output" / "combine_cards"
plots_dir = repo / "output" / "combine_plots"
plots_dir.mkdir(parents=True, exist_ok=True)

if not pkl_path.exists():
    print(f"Missing pickle: {pkl_path}")
    print("Run first: python run_analysis.py --full")
else:
    cmd = [
        "python",
        str(repo / "scripts" / "make_combine_datacards.py"),
        "--input", str(pkl_path),
        "--output-dir", str(cards_dir),
        "--hist-key", "recoil",
        "--region", "sr",
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

    run_all = cards_dir / "run_combine_all.sh"
    print("\nNext commands (in Combine/CMSSW shell):")
    print(f"  bash {run_all}")
    print(
        "  python "
        f"{repo / 'scripts' / 'plot_combine_limits.py'} "
        "--combine-dir "
        f"{cards_dir} "
        "--output "
        f"{plots_dir / 'limits_all_masspoints.png'}"
    )

---
## 6. Summary

- **Weights:** MC event weights (e.g. genWeight) are used when filling histograms; systematics are encoded as normalisation or shape variations.
- **Fit:** Binned Poisson likelihood fit of signal + backgrounds to data; minimisation gives best-fit parameters (e.g. signal strength \(\mu\)).
- **Goodness of fit:** \(\chi^2\), p-value, and pulls quantify how well the model describes the data.
- **Limit:** Upper limit on signal (e.g. 95% CL) via asymptotic formula or pyhf.

**Further reading:** pyhf documentation, TRExFitter, Combine (CMS).

### Exercises

1. Add a second systematic (e.g. shape: scale MET by 1.02 for "up") and re-fit.
2. Compute the limit for a different signal hypothesis (e.g. different cross-section).
3. (If time) Compare asymptotic limit with a toy Monte Carlo limit.